In [1]:
import pandas as pd
import numpy as np

print("Kernel aktif")
print("Pandas:", pd.__version__)
print("Numpy:", np.__version__)

Kernel aktif
Pandas: 3.0.3
Numpy: 1.26.4


In [2]:
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

In [3]:
orders = pd.read_csv(RAW_PATH + "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_PATH + "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_PATH + "olist_order_payments_dataset.csv")
customers = pd.read_csv(RAW_PATH + "olist_customers_dataset.csv")
products = pd.read_csv(RAW_PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_PATH + "olist_sellers_dataset.csv")
reviews = pd.read_csv(RAW_PATH + "olist_order_reviews_dataset.csv")
category_translation = pd.read_csv(RAW_PATH + "product_category_name_translation.csv")
geolocation = pd.read_csv(RAW_PATH + "olist_geolocation_dataset.csv")

print("Semua dataset berhasil diload!")

Semua dataset berhasil diload!


In [4]:
datasets = {
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "customers": customers,
    "products": products,
    "sellers": sellers,
    "reviews": reviews,
    "category_translation": category_translation,
    "geolocation": geolocation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
customers: (99441, 5)
products: (32951, 9)
sellers: (3095, 4)
reviews: (99224, 7)
category_translation: (71, 2)
geolocation: (1000163, 5)


In [5]:
missing_summary = []

for name, df in datasets.items():
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    
    temp = pd.DataFrame({
        "dataset": name,
        "column": missing_count.index,
        "missing_count": missing_count.values,
        "missing_percent": missing_percent.values
    })
    
    missing_summary.append(temp)

missing_summary = pd.concat(missing_summary, ignore_index=True)
missing_summary = missing_summary[missing_summary["missing_count"] > 0]
missing_summary.sort_values(by="missing_percent", ascending=False)

,dataset,column,missing_count,missing_percent
41,reviews,review_comment_title,87656,88.341530
42,reviews,review_comment_message,58247,58.702532
6,orders,order_delivered_customer_date,2965,2.981668
26,products,product_category_name,610,1.851234
27,products,product_name_lenght,610,1.851234
28,products,product_description_lenght,610,1.851234
29,products,product_photos_qty,610,1.851234
5,orders,order_delivered_carrier_date,1783,1.793023
4,orders,order_approved_at,160,0.160899
30,products,product_weight_g,2,0.006070


In [6]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "dataset": name,
        "duplicate_rows": df.duplicated().sum()
    })

pd.DataFrame(duplicate_summary)

,dataset,duplicate_rows
0,orders,0
1,order_items,0
2,payments,0
3,customers,0
4,products,0
5,sellers,0
6,reviews,0
7,category_translation,0
8,geolocation,261831


In [7]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [8]:
# Merge orders dengan customers
df = orders.merge(customers, on="customer_id", how="left")

# Merge order_items dengan products
order_product = order_items.merge(products, on="product_id", how="left")

# Tambahkan nama kategori produk dalam bahasa Inggris
order_product = order_product.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

# Tambahkan data seller
order_product = order_product.merge(sellers, on="seller_id", how="left")

# Merge ke dataset utama
df = df.merge(order_product, on="order_id", how="left")

# Agregasi payment per order
payment_agg = payments.groupby("order_id", as_index=False).agg({
    "payment_value": "sum",
    "payment_installments": "mean"
})

# Ambil payment type yang paling sering muncul per order
payment_type = payments.groupby("order_id")["payment_type"].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
).reset_index()

payment_agg = payment_agg.merge(payment_type, on="order_id", how="left")

# Merge payment ke dataset utama
df = df.merge(payment_agg, on="order_id", how="left")

# Agregasi review score per order
review_agg = reviews.groupby("order_id", as_index=False).agg({
    "review_score": "mean"
})

# Merge review ke dataset utama
df = df.merge(review_agg, on="order_id", how="left")

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,38.71,1.0,voucher,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,141.46,1.0,boleto,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,179.12,3.0,credit_card,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,72.20,1.0,credit_card,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,28.62,1.0,credit_card,5.0


In [9]:
df["order_year"] = df["order_purchase_timestamp"].dt.year
df["order_month"] = df["order_purchase_timestamp"].dt.month
df["order_month_name"] = df["order_purchase_timestamp"].dt.month_name()
df["order_year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

df["delivery_time_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.days

df["is_late_delivery"] = np.where(
    df["order_delivered_customer_date"] > df["order_estimated_delivery_date"],
    1,
    0
)

df["is_delivered"] = np.where(df["order_status"] == "delivered", 1, 0)

df[[
    "order_id",
    "order_status",
    "order_year_month",
    "payment_value",
    "review_score",
    "delivery_time_days",
    "estimated_delivery_days",
    "is_late_delivery",
    "is_delivered"
]].head()

,order_id,order_status,order_year_month,payment_value,review_score,delivery_time_days,estimated_delivery_days,is_late_delivery,is_delivered
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10,38.71,4.0,8.0,15,0,1
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07,141.46,4.0,13.0,19,0,1
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08,179.12,5.0,9.0,26,0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11,72.20,5.0,13.0,26,0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02,28.62,5.0,2.0,12,0,1


In [10]:
print("Final dataset shape:", df.shape)
print("\nJumlah kolom:", len(df.columns))
print("\nDaftar kolom:")
print(df.columns.tolist())

Final dataset shape: (113425, 42)

Jumlah kolom: 42

Daftar kolom:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_value', 'payment_installments', 'payment_type', 'review_score', 'order_year', 'order_month', 'order_month_name', 'order_year_month', 'delivery_time_days', 'estimated_delivery_days', 'is_late_delivery', 'is_delivered']


In [11]:
df.to_csv(PROCESSED_PATH + "ecommerce_main_dataset.csv", index=False)

print("Processed dataset saved successfully!")
print("File saved to:", PROCESSED_PATH + "ecommerce_main_dataset.csv")
print("Shape:", df.shape)

Processed dataset saved successfully!
File saved to: ../data/processed/ecommerce_main_dataset.csv
Shape: (113425, 42)
